# Kết Luận Dự Án: Dự Đoán Khả Năng Bán Chạy Sản Phẩm Trên Tiki.vn

## 📊 Tổng Quan Dự Án

Dự án này thực hiện phân tích toàn diện cho việc mô hình hóa dự đoán khả năng bán chạy của sản phẩm trên sàn thương mại điện tử Tiki.vn. Quy trình bao gồm:

1. **Khảo sát tính khả thi** (crawl.ipynb)
2. **Làm sạch dữ liệu** (data_cleaning.ipynb)
3. **Kỹ thuật đặc trưng và mã hóa** (encode.ipynb)
4. **Trực quan hóa so sánh** (visualization_comparison.ipynb)
5. **Phân tích đa biến** (multivariable_visualization.ipynb)

---

## 1️⃣ Vấn Đề Nghiên Cứu

### Câu Hỏi Chính
**Các đặc trưng của sản phẩm (giá, chiết khấu, đánh giá, số review, miễn phí vận chuyển, danh mục) có thể dự đoán được khả năng bán chạy của sản phẩm trên Tiki.vn hay không?**

### Biến Mục Tiêu (Y)
- **Biến liên tục**: `sold` (số lượng sản phẩm đã bán)
- **Biến phân lớp** (được khuyến cáo): Phân chia thành 2 nhóm
  - **Nhãn 1**: Bán chạy (sold > Q3 hoặc sold > μ + σ)
  - **Nhãn 0**: Bán chậm/tầm thường (sold ≤ ngưỡng)

### Biến Độc Lập (X) - 7 Đặc Trưng

| Biến | Kiểu | Mô Tả |
|------|------|-------|
| `price` | Số | Giá bán (VNĐ) |
| `discount_percent` | Số | Mức giảm giá (%) |
| `product_rating` | Số | Đánh giá từ 0-5 sao |
| `number_of_reviews` | Số | Số lượng bình luận |
| `freeship` | Nhị phân | Miễn phí vận chuyển (0/1) |
| `category` | Danh mục | 31 loại sản phẩm |
| `product_name` | Text | Tên sản phẩm |

---

## 2️⃣ Quy Trình Xử Lý Dữ Liệu

### 📥 Làm Sạch Dữ Liệu (Data Cleaning)
**13 bước xử lý được áp dụng**:

| Bước | Tác Vụ | Phương Pháp |
|------|--------|------------|
| 1-4 | Xử lý cột số (sold, price, discount) | Trích xuất, loại bỏ ký tự đặc biệt, chuyển kiểu |
| 5-7 | Xử lý tên sản phẩm, freeship, category | Chuẩn hóa format, loại bỏ dấu Việt |
| 8-9 | Xử lý rating, number_of_reviews | Điền giá trị thiếu bằng median |
| 10-11 | Xóa dòng không hợp lệ & trùng lặp | Filter data invalid & check (product_name, price, category) |
| 12-13 | Chuẩn hóa kiểu dữ liệu & lưu | Đảm bảo đúng dtype, lưu clean1.csv |

**Kết quả**: Dataset sạch, loại bỏ outlier, sẵn sàng cho phân tích.

### 🔄 Kỹ Thuật Đặc Trưng & Mã Hóa (Feature Engineering)

**Áp dụng 3 bước biến đổi**:

1. **Numeric Scaling**:
   - Đối với biến lệch phải (`price`, `number_of_reviews`, `sold`): Log transform + Robust scaling
   - Đối với biến hữu hạn (`discount_percent`, `product_rating`): Min-Max normalization
   - `freeship`: Giữ nguyên (nhị phân 0/1)

2. **Category One-Hot Encoding**:
   - `category`: Mã hóa 31 danh mục thành vector nhị phân

3. **Text Feature Extraction**:
   - `product_name`: TF-IDF (unigram + bigram), giữ tiếng Việt có dấu

**Output**: clean_encoded.csv - sẵn sàng cho machine learning

---

## 3️⃣ Phát Hiện & Phân Tích Chính

### 📈 Đặc Điểm Phân Phối Dữ Liệu

| Biến | Phân Phối | Nhận Xét |
|------|-----------|----------|
| `price` | Cân bằng tốt | Giá tập trung quanh giá trị trung bình |
| `discount_percent` | Lệch trái | Tập trung 20-40%, có outlier cao |
| `product_rating` | Lệch phải | Hầu hết ≥ 4 sao, khó phân biệt |
| `number_of_reviews` | Lệch trái mạnh | Rất nhiều outlier, ít sản phẩm có review cao |
| `freeship` | Cực đoan | ~100% sản phẩm có free shipping |
| `sold` | Lệch trái | Phần lớn ít bán, một số bán rất nhiều |

**Kết luận**: Dữ liệu có tính chất phân hóa cao - chỉ một số ít sản phẩm chiếm phần lớn lượng tiêu thụ.

### 🔗 Ma Trận Tương Quan Chính

Các mối quan hệ giữa biến độc lập và `sold`:

| Tương Quan với `sold` | Hệ Số | Mức Độ | Diễn Giải |
|----------------------|-------|--------|----------|
| `number_of_reviews` ↔ `sold` | **0.76** | ⭐⭐⭐⭐⭐ Rất mạnh | Sản phẩm bán chạy → nhiều review. Yếu tố dự báo tốt nhất! |
| `discount_percent` ↔ `sold` | **0.34** | ⭐⭐⭐ Trung bình | Giảm giá kích thích mua nhưng không quyết định |
| `product_rating` ↔ `sold` | **0.28** | ⭐⭐⭐ Yếu-trung bình | Rating có ảnh hưởng nhưng không mạnh |
| `price` ↔ `sold` | **-0.13** | ⭐ Rất yếu (âm) | Giá gần như không ảnh hưởng rõ |
| `freeship` ↔ `sold` | **≈ 0** | ❌ Không | Hầu hết có free shipping nên không phân biệt |

---

## 4️⃣ Khả Thi Dự Án

### ✅ Yếu Tố Tích Cực

1. **Tương Quan Logic Mạnh**: `number_of_reviews` có r = 0.76 với `sold` - bảo đảm khả năng dự đoán
2. **Đa Dạng Đặc Trưng**: Kết hợp biến số, danh mục, text - cho phép áp dụng nhiều thuật toán
3. **Dữ Liệu Đủ**: ~1,500+ mẫu / 7 biến = 214:1 (yêu cầu ≥ 10:1) ✓
4. **Thu Thập Dễ**: Dữ liệu từ Selenium scraping, có thể mở rộng thêm
5. **Bài Toán Rõ Ràng**: Phân lớp nhị phân dễ diễn giải cho business

### ⚠️ Thách Thức

1. **Mất Cân Bằng Lớp**: Chỉ một số ít sản phẩm bán chạy (skewed distribution)
   - **Giải pháp**: SMOTE, weighted loss, threshold tuning

2. **Biến `freeship` ít Hữu Ích**: 99% = 1 (hầu như không phân biệt)
   - **Giải pháp**: Có thể loại bỏ hoặc giữ lại để tái kiểm tra



## 5️⃣ Đặc Trưng Quan Trọng Theo Thứ Tự

### 🏆 Xếp Hạng Tầm Quan Trọng

| Xếp Hạng | Biến | Mức Độ | Giải Thích |
|----------|------|--------|----------|
| 1️⃣ | `number_of_reviews` | ⭐⭐⭐⭐⭐ | Tương quan r=0.76, là yếu tố dự báo tốt nhất |
| 2️⃣ | `discount_percent` | ⭐⭐⭐⭐ | Tương quan r=0.34, kích thích nhu cầu mua |
| 3️⃣ | `product_rating` | ⭐⭐⭐ | Tương quan r=0.28, ảnh hưởng vừa phải |
| 4️⃣ | `category` | ⭐⭐⭐ | Giúp mô hình hiểu đặc thù từng ngành |
| 5️⃣ | `price` | ⭐⭐ | Tương quan r=-0.13, thông tin yếu |
| 6️⃣ | `freeship` | ⭐ | Hầu như không phân biệt (tất cả ≈ 1) |
| 7️⃣ | `product_name` | ⭐ | Tùy chọn, NLP phức tạp |

---

## 6️⃣ Các Thuật Toán Đề Xuất

### 🎯 Mô Hình Phân Lớp (Classification)

#### Baseline (Đơn Giản & Nhanh):
- **Logistic Regression**
  - Ưu: Nhanh, dễ diễn giải
  - Nhược: Hiệu suất có thể thấp

#### Chính (Cân Bằng Tốt):
- **Random Forest** (Khuyến cáo)
  - Ưu: Hiệu suất cao, xử lý non-linear, feature importance
  - Nhược: Chậm hơn, ít diễn giải

- **Gradient Boosting (LightGBM/XGBoost)**
  - Ưu: Hiệu suất rất cao, nhanh
  - Nhược: Dễ overfitting

#### Nâng Cao (Nếu Có Thời Gian):
- **Neural Networks**
  - Phù hợp nếu có TF-IDF features từ product_name

### 📊 Mục Tiêu Hiệu Suất

```
Accuracy  ≥ 75%   (tỷ lệ dự đoán đúng)
Precision ≥ 70%   (trong số đoán "bán chạy", bao nhiêu đúng)
Recall    ≥ 70%   (trong thực tế bán chạy, bao nhiêu đã phát hiện)
F1-Score  ≥ 70%   (cân bằng Precision-Recall)
ROC-AUC   ≥ 0.78  (khả năng phân biệt)
```

---

## 7️⃣ Kết Luận Cuối Cùng

### ✅ Dự Án **HOÀN TOÀN KHẢ THI**

**Lý Do**:
1. ✓ Biến mục tiêu `sold` có thể được dự báo dựa trên 7 đặc trưng độc lập
2. ✓ Dữ liệu đã sau qua quy trình làm sạch, xử lý, mã hóa toàn diện
3. ✓ Tương quan giữa `number_of_reviews` và `sold` rất mạnh (r=0.76), đảm bảo khả năng học
4. ✓ Dữ liệu đủ lượng: 1,500+ mẫu > yêu cầu tối thiểu
5. ✓ Bài toán phân lớp nhị phân dễ diễn giải cho ứng dụng thực tiễn
6. ✓ Có đủ loại đặc trưng (số, danh mục, text) để thử nghiệm nhiều thuật toán

### 🎯 Hướng Đi Tiếp Theo

1. **Chia tập dữ liệu**: 70% training, 15% validation, 15% test
2. **Xử lý mất cân bằng lớp**: Áp dụng SMOTE hoặc weighted sampling
3. **Huấn luyện mô hình**: Bắt đầu từ Random Forest, so sánh với XGBoost
4. **Hyper-parameter tuning**: Sử dụng GridSearch hoặc Bayesian Optimization
5. **Đánh giá & Giải Thích**: Feature importance, confusion matrix, ROC curve
6. **Deployment**: Đưa mô hình vào production để hỗ trợ quyết định kinh doanh

### 💡 Giá Trị Kinh Doanh

- **Xác định sản phẩm tiềm năng**: Hỗ trợ chọn hàng hóa sẽ bán chạy
- **Tối ưu hóa danh mục**: Loại bỏ sản phẩm khó bán
- **Marketing thông minh**: Tập trung nguồn lực vào sản phẩm có khả năng cao
- **Dự báo doanh số**: Cải thiện lên kế tài chính

---

## 📋 Tóm Tắt Dữ Liệu

| Thông Tin | Giá Trị |
|-----------|--------|
| **Số dòng dữ liệu** | ~1,500+ sản phẩm |
| **Số biến** | 7 (6 độc lập + 1 mục tiêu) |
| **Loại bài toán** | Phân lớp nhị phân (khuyến cáo) / Hồi quy (phụ) |
| **Tỷ lệ mẫu/biến** | 214:1 (tuyệt vời, yêu cầu ≥10:1) |
| **Biến dự báo tốt nhất** | `number_of_reviews` (r=0.76) |
| **Thách thức** | Mất cân bằng lớp, `freeship` không hữu ích |
| **Mục tiêu hiệu suất** | Accuracy ≥75%, F1-score ≥70% |
| **Thời gian phát triển** | 2-3 tuần |
| **Trạng thái** | ✅ Đủ điều kiện để mô hình hóa |

---

## 🔗 Tài Liệu Tham Khảo

- **crawl.ipynb**: Phát biểu bài toán & khảo sát tính khả thi
- **data_cleaning.ipynb**: Quy trình làm sạch 13 bước
- **encode.ipynb**: Kỹ thuật đặc trưng & mã hóa
- **visualization_comparison.ipynb**: Trực quan hóa so sánh trước/sau xử lý
- **multivariable_visualization.ipynb**: Phân tích đa biến & ma trận tương quan

---

**Kết luận**: Dự án khảo sát này đã hoàn thiện phân tích dữ liệu toàn diện và chứng minh tính khả thi của việc xây dựng mô hình dự đoán khả năng bán chạy. Các bước tiếp theo là huấn luyện và triển khai mô hình học máy.